# M4.4 · LSTM ve GRU

Kapılı hücrelerle uzun bellek. Bir LSTM ve bir GRU adımını elle hesaplayıp hücre durumunun gradyanı neden taşıdığını görüyor, saf Python'da iki hücreyi PyTorch'un formülleriyle yazıyor, NumPy'da LSTM adımının ileri/geri geçişini kuruyor, PyTorch'ta `nn.LSTMCell` ile aynı sayıları veren kendi hücreni yazıyor ve en sonda farklı uzunluktaki dizileri maskeleyerek tek batch'te işliyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-4-lstm-gru.html](https://m-gonul.github.io/pytorch-lab/m4-4-lstm-gru.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_4_lstm_gru.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need\n\n\ndef sig(v):\n    return 1 / (1 + np.exp(-v))\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    rng = np.random.default_rng(0)\n    D, H = 3, 4\n    x, h, c = rng.normal(size=D), rng.normal(size=H) * 0.5, rng.normal(size=H)\n\n    def ls():\n        Wi, Wh, bi, bh = rng.normal(size=(4 * H, D)), rng.normal(size=(4 * H, H)), rng.normal(size=4 * H), rng.normal(size=4 * H)\n        a = Wi @ x + bi + Wh @ h + bh\n        i, f, g, o = sig(a[:H]), sig(a[H:2 * H]), np.tanh(a[2 * H:3 * H]), sig(a[3 * H:])\n        cn = f * c + i * g\n        hn, cc = need(ns, \"lstm_step\")(x.tolist(), h.tolist(), c.tolist(), Wi.tolist(), Wh.tolist(), bi.tolist(), bh.tolist())\n        close(cc, cn, what=\"c\"); close(hn, o * np.tanh(cn), what=\"h\")\n    R.test(\"lstm_step (kapı sırası i, f, g, o)\", ls)\n\n    def gr():\n        Wi, Wh, bi, bh = rng.normal(size=(3 * H, D)), rng.normal(size=(3 * H, H)), rng.normal(size=3 * H), rng.normal(size=3 * H)\n        gi, gh = Wi @ x + bi, Wh @ h + bh\n        r, z = sig(gi[:H] + gh[:H]), sig(gi[H:2 * H] + gh[H:2 * H])\n        n = np.tanh(gi[2 * H:] + r * gh[2 * H:])\n        close(need(ns, \"gru_step\")(x.tolist(), h.tolist(), Wi.tolist(), Wh.tolist(), bi.tolist(), bh.tolist()), (1 - z) * n + z * h)\n    R.test(\"gru_step (r yalnız W_hn·h + b_hn'yi çarpar)\", gr)\n\n    def hand():\n        h1, c1 = need(ns, \"lstm_step\")([0.0], [0.0], [0.5], [[0.0], [0.0], [0.0], [0.0]], [[0.0]] * 4, [0.0, 1.0, 1.0, 0.0], [0.0] * 4)\n        close([c1[0], h1[0]], [0.746326, 0.316476], atol=2e-6)\n    R.test(\"elle örnek\", hand)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    N, D, H = 3, 4, 5\n    x, h, c = rng.normal(size=(N, D)), rng.normal(size=(N, H)), rng.normal(size=(N, H))\n    Wx, Wh, b = rng.normal(size=(D, 4 * H)) * 0.5, rng.normal(size=(H, 4 * H)) * 0.5, rng.normal(size=4 * H) * 0.3\n    dh, dc = rng.normal(size=(N, H)), rng.normal(size=(N, H))\n\n    def fw():\n        hn, cn, _ = need(ns, \"lstm_step_forward\")(x, h, c, Wx, Wh, b)\n        a = x @ Wx + h @ Wh + b; s = lambda v: 1 / (1 + np.exp(-v))\n        cc = s(a[:, H:2 * H]) * c + s(a[:, :H]) * np.tanh(a[:, 2 * H:3 * H])\n        close(cn, cc, what=\"c\"); close(hn, s(a[:, 3 * H:]) * np.tanh(cc), what=\"h\")\n    R.test(\"lstm_step_forward\", fw)\n\n    def bw():\n        _, _, cache = need(ns, \"lstm_step_forward\")(x, h, c, Wx, Wh, b)\n        grads = need(ns, \"lstm_step_backward\")(dh, dc, cache)\n        def f():\n            hn, cn, _ = need(ns, \"lstm_step_forward\")(x, h, c, Wx, Wh, b)\n            return (hn * dh).sum() + (cn * dc).sum()\n        e = 1e-6\n        for arr, g, name in zip((x, h, c, Wx, Wh, b), grads, (\"dx\", \"dprev_h\", \"dprev_c\", \"dWx\", \"dWh\", \"db\")):\n            for idx in list(np.ndindex(arr.shape))[::max(1, arr.size // 8)]:\n                old = arr[idx]; arr[idx] = old + e; p = f(); arr[idx] = old - e; m = f(); arr[idx] = old\n                close(g[idx], (p - m) / (2 * e), rtol=1e-5, atol=1e-7, what=f\"{name}{idx}\")\n    R.test(\"lstm_step_backward: 6 gradyan sayısal türevle aynı\", bw)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n\n    def cell():\n        C = need(ns, \"MyLSTMCell\", \"sınıf\")\n        ref = nn.LSTMCell(4, 6).double(); my = C(4, 6).double()\n        my.load_state_dict(ref.state_dict())\n        x = torch.randn(3, 4, generator=g, dtype=torch.float64)\n        st = (torch.randn(3, 6, generator=g, dtype=torch.float64), torch.randn(3, 6, generator=g, dtype=torch.float64))\n        a, b = my(x, st), ref(x, st)\n        close(a[0], b[0], what=\"h\"); close(a[1], b[1], what=\"c\")\n        close(my(x)[0], ref(x)[0], what=\"state = None\")\n    R.test(\"MyLSTMCell = nn.LSTMCell\", cell)\n\n    def seq():\n        C = need(ns, \"MyLSTMCell\", \"sınıf\")\n        lstm = nn.LSTM(4, 6, batch_first=True).double(); my = C(4, 6).double()\n        my.load_state_dict({k.replace(\"_l0\", \"\"): v for k, v in lstm.state_dict().items()})\n        x = torch.randn(2, 9, 4, generator=g, dtype=torch.float64)\n        out, (h, c) = need(ns, \"run_cell\")(my, x)\n        o2, (h2, c2) = lstm(x)\n        close(out, o2, what=\"out\"); close(h, h2[0], what=\"h_T\"); close(c, c2[0], what=\"c_T\")\n        w = torch.randn_like(out)\n        (out * w).sum().backward(); (o2 * w).sum().backward()\n        close(my.weight_hh.grad, lstm.weight_hh_l0.grad, what=\"∂W_hh\")\n    R.test(\"run_cell = nn.LSTM (dizi + gradyan)\", seq)\n",
 "e": "import torch\nfrom torch import nn\nfrom torch.nn.utils.rnn import pack_padded_sequence\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız zaman döngüsü\", lambda: max_loops(src, 1, func=\"masked_lstm\", why=\"diziler üzerinde döngü olmamalı\"))\n    g = torch.Generator().manual_seed(0)\n    lstm = nn.LSTM(3, 5, batch_first=True).double()\n    P = [lstm.weight_ih_l0.detach(), lstm.weight_hh_l0.detach(), lstm.bias_ih_l0.detach(), lstm.bias_hh_l0.detach()]\n    lengths = torch.tensor([7, 2, 5, 1, 7])\n    x = torch.randn(5, 7, 3, generator=g, dtype=torch.float64)\n    for k, L in enumerate(lengths.tolist()):\n        x[k, L:] = 99.0                                     # dolgu: sızarsa sonucu bozar\n\n    def each():\n        h, c = need(ns, \"masked_lstm\")(x, lengths, *P)\n        for k, L in enumerate(lengths.tolist()):\n            _, (hk, ck) = lstm(x[k:k + 1, :L])\n            close(h[k], hk[0, 0], what=f\"dizi {k} h\"); close(c[k], ck[0, 0], what=f\"dizi {k} c\")\n    R.test(\"her dizi tek başına çalıştırılmış gibi (dolgu sızmıyor)\", each)\n\n    def packed():\n        h, c = need(ns, \"masked_lstm\")(x, lengths, *P)\n        _, (h2, c2) = lstm(pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False))\n        close(h, h2[0]); close(c, c2[0])\n    R.test(\"pack_padded_sequence + nn.LSTM ile aynı\", packed)\n    if \"masked_lstm\" not in ns:\n        return\n    lb = nn.LSTM(32, 64, batch_first=True)\n    Pb = [p.detach() for p in (lb.weight_ih_l0, lb.weight_hh_l0, lb.bias_ih_l0, lb.bias_hh_l0)]\n    Lb = torch.randint(10, 100, (64,), generator=g)\n    xb = torch.randn(64, 100, 32, generator=g)\n    with torch.no_grad():\n        t_loop = bench(lambda: [lb(xb[k:k + 1, :int(Lb[k])]) for k in range(64)], repeat=3, number=1)\n        t_vec = bench(lambda: ns[\"masked_lstm\"](xb, Lb, *Pb), repeat=3, number=1)\n    R.table(\"64 dizi, uzunluk 10–100\", [\"yöntem\", \"süre\", \"kazanç\"],\n            [[\"for dizi: nn.LSTM\", fmt_time(t_loop), \"1×\"], [\"senin masked_lstm (batch + maske)\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}×\"]],\n            note=\"Maskeli batch en uzun dizinin uzunluğu kadar adım atar; kısa dizilerin dolgusu boşa hesaplanır. Uzunluğa göre gruplama (bucketing) bunu azaltır.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m4_4_lstm_gru as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

LSTM, RNN'in gizli durumuna ek olarak bir **hücre durumu** $c_t$ taşır ve onu kapılarla günceller. PyTorch'un formülü (kapı sırası **i, f, g, o**):

$$
\begin{aligned}
[a_i,a_f,a_g,a_o]&=W_{ih}x_t+b_{ih}+W_{hh}h_{t-1}+b_{hh}\quad(4H\text{ boyutlu, tek matmul})\\
i&=\sigma(a_i),\quad f=\sigma(a_f),\quad g=\tanh(a_g),\quad o=\sigma(a_o)\\
c_t&=f\odot c_{t-1}+i\odot g,\qquad h_t=o\odot\tanh(c_t).
\end{aligned}
$$

Kritik nokta: $c_t$'den $c_{t-1}$'e giden yol **toplamsal** ve türevi yalnız $f$. Ağırlık matrisi ve doyurucu bir türev içermediği için gradyan uzun mesafe taşınır.

GRU iki kapıyla aynı fikri daha az parametreyle uygular (kapı sırası **r, z, n**):

$$
r=\sigma(W_{ir}x+b_{ir}+W_{hr}h+b_{hr}),\quad z=\sigma(\cdots),\quad n=\tanh\big(W_{in}x+b_{in}+r\odot(W_{hn}h+b_{hn})\big),\quad h'=(1-z)\odot n+z\odot h .
$$

## 1 · Elle

Kapıları hesapla ve hücre durumunu güncelle. Forget kapısı $0.73$: eski belleğin $\%73$'ü korunuyor; input kapısı yeni adaydan yarısını ekliyor.

Gradyan için yalnız $c$ yolunu düşün: $\partial c_t/\partial c_{t-1}=f$. $f=0.9$ ile 10 adım sonra gradyan $0.35$; sade RNN'de (her adımda en fazla $0.25$ çarpanlı sigmoid ya da $\lVert W\rVert<1$ ile tanh) $10^{-6}$ mertebesine iner.

GRU'da $z$ eski durumla yeni aday arasında bir interpolasyon: $z=1$ ise hiçbir şey değişmez.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**LSTM adımı; kapı ön-aktivasyonları $a_i=0$, $a_f=1$, $a_g=1$, $a_o=0$; $c_{t-1}=0.5$**

- `f`: $f=\sigma(a_f)$
- `c`: $c_t=f\,c_{t-1}+i\,g$
- `h`: $h_t=o\tanh(c_t)$
- `dc`: $\partial c_t/\partial c_{t-1}$
- `car`: $f=0.9$ ile 10 adım: $\prod f$

**GRU: $z=0.5$, $h_{t-1}=0.4$, aday $n=\tanh(0.8)$: $h_t=(1-z)\,n+z\,h_{t-1}$**

- `n`: $n$
- `hg`: $h_t$

In [ ]:
cevaplar = dict(
    f=None,
    c=None,
    h=None,
    dc=None,
    car=None,
    n=None,
    hg=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"f":[0.731059,0.002],"c":[0.746326,0.003],"h":[0.316476,0.003],"dc":[0.731059,0.002],"car":[0.348678,0.002],"n":[0.664037,0.002],"hg":[0.532018,0.002]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, tek örnek; matrisler satır listesi. PyTorch düzeni: `W_ih (4H × D)`, `W_hh (4H × H)`, `b_ih`, `b_hh (4H,)`; satır blokları LSTM'de `i, f, g, o`, GRU'da `(3H)` ve `r, z, n`.

- `lstm_step(x, h, c, W_ih, W_hh, b_ih, b_hh)` → `(h_yeni, c_yeni)`
- `gru_step(x, h, W_ih, W_hh, b_ih, b_hh)` → `h_yeni`. $n$'de $r$, yalnız $W_{hn}h+b_{hn}$ kısmını çarpar.

In [ ]:
%%writefile stage_b.py
import math


def lstm_step(x, h, c, W_ih, W_hh, b_ih, b_hh):
    ...


def gru_step(x, h, W_ih, W_hh, b_ih, b_hh):
    ...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, batch'li, satır vektörleri: `Wx (D, 4H)`, `Wh (H, 4H)`, `b (4H,)`, sütun blokları `i, f, g, o`. **Döngü yok.**

- `lstm_step_forward(x, prev_h, prev_c, Wx, Wh, b)` → `(next_h, next_c, cache)`
- `lstm_step_backward(dnext_h, dnext_c, cache)` → `(dx, dprev_h, dprev_c, dWx, dWh, db)`

Geri geçişte dört kapının ön-aktivasyon gradyanlarını `(N, 4H)` bir matrise yan yana koy; sonra tek seferde $dx=da\,W_x^\top$, $dW_x=x^\top da$.

In [ ]:
%%writefile stage_c.py
import numpy as np


def lstm_step_forward(x, prev_h, prev_c, Wx, Wh, b):
    ...


def lstm_step_backward(dnext_h, dnext_c, cache):
    """→ (dx, dprev_h, dprev_c, dWx, dWh, db)"""
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `MyLSTMCell(input_size, hidden_size)` — parametre adları `nn.LSTMCell` ile aynı: `weight_ih (4H, D)`, `weight_hh (4H, H)`, `bias_ih`, `bias_hh`. `forward(x, state=None)` → `(h, c)`; `state` yoksa sıfırlar.
- `run_cell(cell, x)` — `x: (N, T, D)`; hücreyi zamanda çalıştır → `(out: (N, T, H), (h_T, c_T))`.

Kontrol `nn.LSTMCell` ile tek adım eşitliğini, `nn.LSTM` (state_dict'i `_l0` adlarıyla kopyalayarak) ile tüm dizi eşitliğini ve gradyanları denetler.

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn


class MyLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        ...

    def forward(self, x, state=None):
        ...


def run_cell(cell, x):
    """x: (N, T, D) → (out (N, T, H), (h_T, c_T))"""
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Farklı uzunlukta $N$ diziyi tek batch'te işle: `x: (N, T_max, D)` dolgulu, `lengths: (N,)`. Her adımda maske $m_t=[t<\text{len}]$ ile güncellemeyi yalnız geçerli dizilere uygula:

$$
h\leftarrow m_t\odot h_{\text{yeni}}+(1-m_t)\odot h,\qquad c\leftarrow m_t\odot c_{\text{yeni}}+(1-m_t)\odot c .
$$

- `masked_lstm(x, lengths, W_ih, W_hh, b_ih, b_hh)` → `(h_T, c_T)`: her dizinin **kendi** son geçerli adımındaki durum. Yalnız zaman döngüsü (dizi döngüsü yok). Dört kapı tek matmul.

Kontrol: her diziyi tek başına çalıştırmak ve `pack_padded_sequence` + `nn.LSTM` ile aynı sonuç; hız tablosu.

In [ ]:
%%writefile stage_e.py
import torch


def masked_lstm(x, lengths, W_ih, W_hh, b_ih, b_hh):
    """x: (N, T, D), lengths: (N,) → (h_T, c_T) her dizinin kendi son adımında. Yalnız zaman döngüsü."""
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [Hochreiter & Schmidhuber — Long Short-Term Memory](https://www.bioinf.jku.at/publications/older/2604.pdf) — ©
- [Colah — Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) — ©
- [Dive into Deep Learning §10.1–10.2 LSTM, GRU](https://d2l.ai/chapter_recurrent-modern/index.html) — CC BY-SA 4.0 / MIT-0
- [CS231n Assignment 3 — LSTM (lstm_step_forward/backward)](https://cs231n.github.io/assignments2025/assignment3/) — MIT
- [PyTorch — nn.LSTMCell / nn.GRUCell formülleri](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTMCell.html) — BSD-3